## CARLA Exploration

In [7]:
import sys
# Need this for file control so programs can be developed outside of the installation path
f_path = '''c:/Users/mikel/Downloads/Other Important Things/RWTH-Aachen Stuff/
            Indpendent_Study/CARLA_0.9.15/WindowsNoEditor/PythonAPI/carla/dist
            carla-0.9.15-py3.7-win-amd64.egg'''

agent_path = '''c:/Users/mikel/Downloads/Other Important Things/RWTH-Aachen Stuff/
            Indpendent_Study/CARLA_0.9.15/WindowsNoEditor/PythonAPI/carla'''

sys.path.append(f_path)
sys.path.append(agent_path)

import carla
import random
import numpy as np
import time
import cv2

In [16]:
for i, f in enumerate(sys.path):
    if f == '''c:/Users/mikel/Downloads/Other Important Things/RWTH-Aachen Stuff/
            Indpendent_Study/CARLA_0.9.15/WindowsNoEditor/PythonAPI/carla''':
        print(i)

15
17


In [17]:
sys.path[15]

'c:/Users/mikel/Downloads/Other Important Things/RWTH-Aachen Stuff/\n            Indpendent_Study/CARLA_0.9.15/WindowsNoEditor/PythonAPI/carla'

In [ ]:
from sys.path[15].agents.navigation.global_route_planner import GlobalRoutePlanner

ModuleNotFoundError: No module named 'agents'

### Map Initialization

In [5]:
# Connect to the client and retrieve the world object
client = carla.Client("localhost", 2000)
world = client.get_world()

# Get the vehicle objects from the world library
vehicle_bp = world.get_blueprint_library().filter("*vehicle*")

# Get the map's spawn points
spawn_points = world.get_map().get_spawn_points()

### Spawning Vehicles/Testing Methods

In [6]:
# Spawn 20 vehicles randomly distributed throughout the map
for i in range(20):
    vehicle = random.choice(vehicle_bp)
    location = random.choice(spawn_points)
    world.try_spawn_actor(vehicle, location)

In [7]:
# Acquire a test vehicle to observe
test_vehicle = random.choice(world.get_actors().filter("*vehicle*"))

In [9]:
# Get location/rotation before movement begins
test_vehicle_pos = test_vehicle.get_transform()
print(test_vehicle_pos)

Transform(Location(x=102.566200, y=43.965668, z=-0.004616), Rotation(pitch=0.000061, yaw=90.390678, roll=-0.000671))


In [10]:
# Access only location
print(test_vehicle_pos.location)

# Access one dimension
print(test_vehicle_pos.location.x)

Location(x=102.566200, y=43.965668, z=-0.004616)
102.56620025634766


In [200]:
# Define the spectator and set the location by the test vehicle
spectator = world.get_actors()[0]

# If the spectator (me) is not at the test_vehicle location, update this
spectator.set_transform(carla.Transform(
    test_vehicle.get_transform().location + carla.Location(y=13, z=6),
    carla.Rotation(
        pitch=test_vehicle.get_transform().rotation.pitch - 13,
        yaw=test_vehicle.get_transform().rotation.yaw - 180
    )))

### Spawning Sensors and Testing Methods

In [153]:
# any time a modification to the block below (with spawning the camera) is made,
# another object will be spawned. Destory all cameras and then respawn one using
# block below so that only one is active at a time
for actor in world.get_actors():
    if actor.type_id == "sensor.camera.rgb":
        actor.destroy()

In [154]:
# previously, I was calculating the camera position globally, but it needs to be done
# relative to the vehicle so that the 'attach_to' method can work properly
relative_offset = carla.Location(z=1.5)

camera_transform = carla.Transform(relative_offset, test_vehicle.get_transform().rotation)

camera_bp = world.get_blueprint_library().find("sensor.camera.rgb")
# # setting image aspect ratio (not sure if this is necessary, default is 800 x 600)
# camera_bp.set_attribute("image_size_x", "1920")
# camera_bp.set_attribute("image_size_y", "1080")

camera = world.spawn_actor(camera_bp, camera_transform, attach_to=test_vehicle)

In [203]:
# since the camera object doesn't visually render, use this code to verify that 
# it is 
bound_box = carla.BoundingBox(camera.get_transform().location,
                              carla.Vector3D(0.5, 0.3, 0.3))

world.debug.draw_box(bound_box, camera.get_transform().rotation,
                     life_time=10, color=carla.Color(100,0,0))

In [180]:
# alternatively you can see this by each object's transform
for actor in world.get_actors():
    if actor.type_id == "sensor.camera.rgb":
        print(actor.get_transform())

print(test_vehicle.get_transform())

Transform(Location(x=102.566216, y=43.965668, z=1.495384), Rotation(pitch=0.000731, yaw=-179.218628, roll=-0.000605))
Transform(Location(x=102.566200, y=43.965668, z=-0.004616), Rotation(pitch=0.000061, yaw=90.390678, roll=-0.000671))


Maintain speed function (from "Full Sim Driving" YouTube video)

In [68]:
PREFERRED_SPEED = 20
SPEED_THRESHOLD = 2

def maintain_speed(s):
    '''
    Simple, undefined function. Open for additional refinement.
    '''
    if s >= PREFERRED_SPEED:
        return 0
    elif s < PREFERRED_SPEED - SPEED_THRESHOLD:
        return 0.8 # almost like % of "full gas"
    else:
        return 0.4

In [60]:
# Set the vehicles to the autopilot state
for vehicle in world.get_actors().filter('*vehicle*'):
    vehicle.set_autopilot(True)

In [54]:
# Get the location and rotation of a vehicle
print(test_vehicle.get_transform())

Transform(Location(x=93.410988, y=29.619640, z=-0.004365), Rotation(pitch=-0.518309, yaw=11.844669, roll=-1.972412))


In [55]:
# Get the velocity and acceleration of a vehicle
print("Vehicle velocity:", test_vehicle.get_velocity())
print("Vehicle acceleration:", test_vehicle.get_acceleration())

Vehicle velocity: Vector3D(x=0.604757, y=2.032999, z=0.000250)
Vehicle acceleration: Vector3D(x=0.272794, y=3.442598, z=-0.008956)


In [28]:
# Another example for how to get offset a location/rotation relative to an existing one
# Pay attention to the difference in style for location vs. rotation
offset_pos = carla.Transform(spectator.get_transform().location + carla.Location(z=5),
                          carla.Rotation(yaw = spectator.get_transform().rotation.yaw - 90))


In [29]:
print("Spectator position: {}".format(spectator_pos))
print("Offset position: {}".format(offset_pos))

Spectator position: Transform(Location(x=102.566200, y=43.965668, z=2.495384), Rotation(pitch=0.000061, yaw=90.390678, roll=-0.000671))
Offset position: Transform(Location(x=102.978058, y=48.993774, z=7.132627), Rotation(pitch=0.000000, yaw=-184.073303, roll=0.000000))


In [106]:
# create a route given the test vehicle location
# uses carla's built in Global Route Planner
town_map = world.get_map()
sampling_res = 2

grp = GlobalRoutePlanner(town_map, sampling_res)

start_pt = test_vehicle.get_transform().location
end_pt = test_vehicle.get_transform().location
end_pt.y += 20

route = grp.trace_route(start_pt, end_pt)

for waypoint in route:
    world.debug.draw_string(waypoint[0].transform.location, "^",
        color=carla.Color(r=0, g=0, b=255), life_time=120, 
        persistent_lines=True)

In [ ]:
# Destroy all of the vehicles
for vehicle in world.get_actors().filter("*vehicle*"):
    vehicle.destroy()

: 